# Challenger Model Assessment

This notebook reproduces the challenger assessment for the final gasoline price regression. All challengers use the same final feature set, so the comparison isolates the estimator choice.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, LassoCV, LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_mensual_final.csv'
SPLIT_DATE = '2023-01-01'
FINAL_FEATURES = ['brent_lag1', 'fx', 'covid_post', 'brent_lag1_x_covid_post']

In [ ]:
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['month'] + '-01')
df = df.sort_values('date').reset_index(drop=True)
df['covid_post'] = (df['date'] >= '2020-03-01').astype(int)
df['brent_lag1_x_covid_post'] = df['brent_lag1'] * df['covid_post']
df_model = df.dropna(subset=['pvp', *FINAL_FEATURES]).reset_index(drop=True)
print(df_model[['month','pvp', *FINAL_FEATURES]].head())
print('\nEffective sample:', df_model['month'].iloc[0], 'to', df_model['month'].iloc[-1], '| n =', len(df_model))

## Final Regression

The final regression keeps the features selected in the econometric analysis. HC3 robust standard errors are used because the residual diagnostics suggested imperfect classical assumptions.

In [ ]:
X = sm.add_constant(df_model[FINAL_FEATURES])
ols_fit = sm.OLS(df_model['pvp'], X).fit(cov_type='HC3')
print(ols_fit.summary())

## Challenger Models

The models below are compared on a chronological holdout split. Random splitting is avoided because the data are monthly time series observations. The objective is to test whether common alternatives materially improve the selected regression using the same information set.

In [ ]:
train = df_model[df_model['date'] < SPLIT_DATE].copy()
test = df_model[df_model['date'] >= SPLIT_DATE].copy()

models = [
    ('OLS', 'linear benchmark', LinearRegression()),
    ('Ridge', 'regularized linear model', make_pipeline(StandardScaler(), RidgeCV(alphas=[0.01,0.1,1,10,100]))),
    ('Lasso', 'sparse regularized linear model', make_pipeline(StandardScaler(), LassoCV(alphas=[0.001,0.01,0.1,1], cv=5, max_iter=10000))),
    ('Elastic Net', 'mixed regularized linear model', make_pipeline(StandardScaler(), ElasticNetCV(alphas=[0.001,0.01,0.1,1], l1_ratio=[0.2,0.5,0.8], cv=5, max_iter=10000, random_state=42))),
    ('Decision Tree', 'simple non-linear tree', DecisionTreeRegressor(max_depth=3, min_samples_leaf=5, random_state=42)),
    ('Random Forest', 'bagged tree ensemble', RandomForestRegressor(n_estimators=500, max_depth=5, min_samples_leaf=3, random_state=42)),
    ('Gradient Boosting', 'boosted tree ensemble', GradientBoostingRegressor(n_estimators=300, learning_rate=0.03, max_depth=2, min_samples_leaf=3, random_state=42)),
    ('XGBoost', 'boosted tree ensemble', XGBRegressor(n_estimators=150, learning_rate=0.03, max_depth=2, subsample=0.8, colsample_bytree=0.9, objective='reg:squarederror', random_state=42, n_jobs=1)),
    ('SVR', 'kernel method', make_pipeline(StandardScaler(), SVR(C=10, epsilon=0.02))),
]

assessments = {
    'OLS': 'Recommended baseline',
    'Lasso': 'Marginal gain',
    'Ridge': 'Marginal gain',
    'Elastic Net': 'Marginal gain',
    'Gradient Boosting': 'Not selected',
    'SVR': 'Not selected',
    'Random Forest': 'Not selected',
    'XGBoost': 'Not selected',
    'Decision Tree': 'Not selected',
}

rows = []
predictions = []
for name, method_type, estimator in models:
    model = clone(estimator)
    model.fit(train[FINAL_FEATURES], train['pvp'])
    pred = model.predict(test[FINAL_FEATURES])
    rows.append({
        'model': name,
        'method_type': method_type,
        'MAE': mean_absolute_error(test['pvp'], pred),
        'RMSE': math.sqrt(mean_squared_error(test['pvp'], pred)),
        'R2_holdout': r2_score(test['pvp'], pred),
        'Bias': float(np.mean(pred - test['pvp'])),
        'assessment': assessments[name],
    })
    predictions.append(pd.DataFrame({'date': test['date'], 'actual': test['pvp'].values, 'prediction': pred, 'model': name}))

comparison = pd.DataFrame(rows).sort_values(['MAE','RMSE']).reset_index(drop=True)
predictions = pd.concat(predictions, ignore_index=True)
comparison

## Generated Comparison Table

The table below is generated by `src/create_model_comparison_and_selection.py` from the current dataset.

| model             | method_type                     | MAE    | RMSE   | R2_holdout | Bias    | assessment           |
| ----------------- | ------------------------------- | ------ | ------ | ---------- | ------- | -------------------- |
| Lasso             | sparse regularized linear model | 0.0458 | 0.0558 | 0.4668     | -0.0194 | Marginal gain        |
| Ridge             | regularized linear model        | 0.0461 | 0.0561 | 0.4616     | -0.0199 | Marginal gain        |
| Elastic Net       | mixed regularized linear model  | 0.0462 | 0.0561 | 0.4601     | -0.0197 | Marginal gain        |
| OLS               | linear benchmark                | 0.0464 | 0.0563 | 0.4578     | -0.0198 | Recommended baseline |
| Gradient Boosting | boosted tree ensemble           | 0.0652 | 0.0851 | -0.2418    | -0.0425 | Not selected         |
| SVR               | kernel method                   | 0.0751 | 0.0860 | -0.2672    | -0.0581 | Not selected         |
| Random Forest     | bagged tree ensemble            | 0.0784 | 0.0928 | -0.4748    | -0.0611 | Not selected         |
| XGBoost           | boosted tree ensemble           | 0.0853 | 0.1002 | -0.7193    | -0.0776 | Not selected         |
| Decision Tree     | simple non-linear tree          | 0.1093 | 0.1239 | -1.6314    | -0.0959 | Not selected         |

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
plot_df = comparison.sort_values('MAE')
ax.barh(plot_df['model'], plot_df['MAE'])
ax.invert_yaxis()
ax.set_title('Holdout MAE by model')
ax.set_xlabel('MAE, EUR/litre')
ax.grid(axis='x', alpha=0.25)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
selected = predictions[predictions['model'].isin(['OLS', 'Lasso', 'XGBoost'])]
actual = selected[['date', 'actual']].drop_duplicates()
ax.plot(actual['date'], actual['actual'], label='Actual', linewidth=2.5, color='black')
for model_name in selected['model'].unique():
    frame = selected[selected['model'] == model_name]
    ax.plot(frame['date'], frame['prediction'], label=model_name)
ax.set_title('Holdout predictions: selected models')
ax.set_ylabel('EUR/litre')
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## Assessment

The regularized linear models produce slightly lower holdout errors than OLS, but the gain is small. The final OLS regression remains the recommended baseline because it is close to the best predictive alternatives and keeps the coefficients directly explainable. The non-linear models, including XGBoost, do not justify their additional complexity on this sample.